# M8-BGT — Name+Address Race/Ethnicity Probability (with Uncertainty)

**AWS Marketplace SageMaker sample notebook.**

This notebook shows how to use the **M8-BGT** model package after subscribing on AWS
Marketplace. Given a person's **name and mailing address**, the model returns:

- probabilities for 5 race/ethnicity classes: `white, black, hispanic, asian, other`
- `dominant_race` (the argmax class)
- `iu` — a calibrated 0-1 **Index of Uncertainty** (how much to trust the row)
- **address lookup-quality codes** (`lookup_level/conf/status`): block-group -> tract -> ZIP -> name-only
- opaque grouping tokens (`cohort_tract`, `cohort_zcta`) for aggregating by neighborhood
  **without** exposing any Census geography

The model resolves the address **entirely offline inside the container** (no third-party
geocoding calls) and can run with **network isolation**. Raw Census GEOIDs are never returned.

> The sample data used here (`sample_input.csv`) is **fully synthetic** — invented names paired
> with well-known public building addresses. It contains no personal data.


## 1. Prerequisites

1. **Subscribe** to *M8-BGT* on AWS Marketplace. After subscribing, copy the
   **product ARN** shown on the product's *Configuration* page.
2. Run this notebook on a SageMaker notebook instance / Studio with an execution role
   that allows `sagemaker:CreateModel`, endpoint + transform operations, and S3 access.
3. `pip install -U sagemaker boto3 pandas` if needed.


In [ ]:
import sagemaker, boto3, pandas as pd, io
from sagemaker import ModelPackage, get_execution_role
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

sess = sagemaker.Session()
role = get_execution_role()
region = boto3.Session().region_name
bucket = sess.default_bucket()
print('region:', region, '| role:', role.split('/')[-1])


### Set your subscribed product ARN
Replace the value below with the **product ARN from your Marketplace subscription**.
(The example is the seller's model-package ARN; after you subscribe, AWS gives you a
product ARN of the form `arn:aws:sagemaker:<region>:<acct>:model-package/<name>-<hash>`.)


In [ ]:
model_package_map = {
    # region : subscribed product ARN
    'us-east-1': 'arn:aws:sagemaker:us-east-1:966917020084:model-package/m8-bgt-race-v1',
}
assert region in model_package_map, f'Subscribe in {region} and add its product ARN'
model_package_arn = model_package_map[region]
model_package_arn


## 2. Real-time inference
Create a deployable model from the package and stand up an endpoint.
Recommended instance: `ml.m5.xlarge` (or `ml.m5.2xlarge` for ~1.8x throughput).


In [ ]:
model = ModelPackage(role=role, model_package_arn=model_package_arn, sagemaker_session=sess)
endpoint_name = 'm8-bgt-demo'
predictor = model.deploy(initial_instance_count=1,
                         instance_type='ml.m5.xlarge',
                         endpoint_name=endpoint_name,
                         serializer=CSVSerializer(),
                         deserializer=CSVDeserializer())
print('endpoint ready:', endpoint_name)


In [ ]:
# Load the synthetic sample and score it via the real-time endpoint.
df_in = pd.read_csv('sample_input.csv', dtype=str)
csv_bytes = df_in.to_csv(index=False)
rows = predictor.predict(csv_bytes)              # list-of-lists (CSV)
out = pd.DataFrame(rows[1:], columns=rows[0])
out.head(15)


In [ ]:
# The probability + diagnostic columns:
cols = ['fname','lname','dominant_race','p_white','p_black','p_hispanic','p_asian','p_other',
        'iu','lookup_level','lookup_status','cohort_tract']
out[cols]


Notice: well-resolved addresses score at `lookup_level=BG` (`lookup_status=ok`), coarser
matches fall back to `TRACT`/`ZIP`, and the unresolvable row (`A015`) is honestly demoted to
`NONE` / `geo_total_failure` and scored on name alone.

**`iu` and `lookup_*` are independent signals.** `iu` measures the model's uncertainty about
the *prediction itself* — not the address-lookup precision. A name-only row with a highly
diagnostic surname can have **low** `iu`, while a fully block-group-resolved row whose name and
neighborhood signals conflict can have **high** `iu`. Use `lookup_*` to filter on address-data
quality and `iu` to filter on prediction confidence.


## 3. Batch transform (score a file)
For large jobs, run a batch transform instead of a live endpoint. Billing is per
transform-instance-hour; throughput is ~450k rows/hr on `ml.m5.xlarge`.


In [ ]:
input_s3 = sess.upload_data('sample_input.csv', bucket=bucket, key_prefix='m8bgt/demo/input')
transformer = model.transformer(instance_count=1, instance_type='ml.m5.xlarge',
                                output_path=f's3://{bucket}/m8bgt/demo/output',
                                accept='text/csv')
transformer.transform(input_s3, content_type='text/csv', split_type='None')
transformer.wait()
print('batch output:', transformer.output_path)


In [ ]:
# Read the batch result back.
s3 = boto3.client('s3')
out_uri = transformer.output_path + '/sample_input.csv.out'
b = out_uri.replace('s3://','').split('/',1)
body = s3.get_object(Bucket=b[0], Key=b[1])['Body'].read().decode()
pd.read_csv(io.StringIO(body)).head(15)


## 4. Output schema

| column | meaning |
|---|---|
| `p_white,p_black,p_hispanic,p_asian,p_other` | class probabilities (sum ~ 1) |
| `dominant_race` | argmax class label |
| `iu` | calibrated Index of Uncertainty, 0 (confident) -> 1 (uncertain) |
| `lookup_level` | geo precision used: `BG` / `TRACT` / `ZIP` / `NONE` |
| `lookup_conf` | nominal confidence of that level (0.97 / 0.75 / 0.40 / 0.00) |
| `lookup_status` | `ok` / `tract_low_conf` / `zip_only` / `geo_total_failure` |
| `cohort_tract`, `cohort_zcta` | opaque grouping tokens (empty when not resolvable) |
| `model_version`, `geo_version` | frozen provenance stamps |

All of your input columns are passed through unchanged. Raw Census GEOIDs are never returned.


## 5. Clean up
Delete the endpoint so you stop incurring charges.


In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)
print('endpoint deleted')
